# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahadumar/flyrank-MLinternship-Assignment-1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"CSV not found — working dir: {os.getcwd()}"

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Working directory: {os.getcwd()}")
print("Ready.")

Loaded: 30,000 rows, 44 columns
Working directory: /content/flyrank-ml-internship-starter
Ready.


## 1. My lane as an ML task (type)

**Task type: Scoring → Ranking**

Lane 2 is a **scoring and ranking** problem, not a binary classification problem.
The distinction matters and is worth explaining carefully.

A classification model answers: *"Is this page declining — yes or no?"*
That is useful, but it is not the decision a content team actually needs to make.
A content team does not have unlimited writer time. They need to know: *"Of all
the pages that need attention, which ones do I work on first?"* That requires an
ordered list — a ranked queue — not a binary label.

The task is therefore:
- **Assign each page a continuous refresh opportunity score** (0 to 1)
- **Rank all pages by that score**, highest to lowest
- **Output a prioritized queue** with reason codes explaining each score

This maps to what the ML literature calls a **pointwise ranking** approach:
train a model to predict a score per page, then sort by score. The evaluation
metric (Precision@K) directly measures ranking quality — how many of the top K
pages the model surfaces are genuinely high-priority.

This is better than pure classification here because:
1. A binary yes/no doesn't tell you *how urgent* a page's need is
2. The content team's capacity constraint (e.g. review top 20 pages per week)
   makes ranked output directly actionable
3. Precision@K is a natural business metric — it measures exactly what the
   team cares about: are the pages we're flagging actually worth fixing?

In [2]:
# Show the distribution of trend directions to confirm this is a ranking problem
# If everything were clearly declining or not, a simple rule would suffice.
# The mix of states is what makes ranking necessary.

trend_counts = df["trend_direction"].value_counts()
trend_pct = (trend_counts / len(df) * 100).round(1)

print("Distribution of page trend states:")
print("=" * 40)
for direction, count in trend_counts.items():
    print(f"  {direction:<10} {count:>6,} pages  ({trend_pct[direction]}%)")

print()
print(f"Total pages: {len(df):,}")
print()
print("Key insight: multiple trend states exist simultaneously.")
print("A binary rule cannot distinguish urgency WITHIN the 'down' group.")
print("A scoring model can — that is why ranking is the right task type.")

Distribution of page trend states:
  down       16,262 pages  (54.2%)
  stable      5,962 pages  (19.9%)
  up          4,388 pages  (14.6%)
  new         2,236 pages  (7.5%)
  flat        1,152 pages  (3.8%)

Total pages: 30,000

Key insight: multiple trend states exist simultaneously.
A binary rule cannot distinguish urgency WITHIN the 'down' group.
A scoring model can — that is why ranking is the right task type.


## 2. Target or proxy

**Ideal outcome:** "This page needs a content refresh."

There is no column in the dataset called `needs_refresh`. No human labeled
these pages. So we need a **proxy label** — a column that is a reasonable
stand-in for refresh need.

**Proxy used:** `trend_direction == "down"` → label = 1 (refresh candidate)
All other trend states (up, stable, flat, new) → label = 0

**Why this proxy is reasonable:**
A page whose impressions or clicks are trending downward is a strong signal
that its content may be losing relevance, freshness, or competitive position.
Refreshing such pages is the core content maintenance action in SEO workflows.

**Why this proxy is imperfect (honest framing):**
- A page can decline for seasonal reasons unrelated to content quality
- A page can be "down" after a recent spike — temporary regression, not decay
- A page that is stable but was never good is missed entirely
- "Down" is itself derived from `trend_pct`, which is a 90-day trailing metric

These limitations do not invalidate the proxy. They define the boundaries of
what the model can honestly claim. The model will say: *"These pages show
signals associated with declining trend in this dataset."* It will not say:
*"These pages need to be refreshed."*

**Label definition (exact):**
```python
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)
```

**Base rate:** 54.2% of pages carry label = 1. This is a moderately imbalanced
problem — the model needs `class_weight="balanced"` or equivalent to avoid
predicting the majority class by default.

In [7]:
# Define the proxy label and show its distribution
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

base_rate = df["is_declining"].mean()
n_declining = df["is_declining"].sum()
n_not = len(df) - n_declining

print("Proxy label: is_declining (trend_direction == 'down')")
print("=" * 50)
print(f"  Label = 1 (declining):     {n_declining:,} pages  ({base_rate*100:.1f}%)")
print(f"  Label = 0 (not declining): {n_not:,} pages  ({(1-base_rate)*100:.1f}%)")
print()
print("Why trend_pct must be EXCLUDED (leakage check):")
print("-" * 50)

# Demonstrate leakage risk — trend_pct is the numeric version of the label
corr = df["trend_pct"].corr(df["is_declining"])
print(f"  Correlation (Pearson): {corr:.3f}")
print(f"  → Moderate negative correlation expected: trend_pct is continuous,")
print(f"    is_declining is binary. But trend_pct is derived from the same")
print(f"    underlying signal as the label — including it would be leakage.")

Proxy label: is_declining (trend_direction == 'down')
  Label = 1 (declining):     16,262 pages  (54.2%)
  Label = 0 (not declining): 13,738 pages  (45.8%)

Why trend_pct must be EXCLUDED (leakage check):
--------------------------------------------------
  Correlation (Pearson): -0.141
  → Moderate negative correlation expected: trend_pct is continuous,
    is_declining is binary. But trend_pct is derived from the same
    underlying signal as the label — including it would be leakage.


## 3. Success metric

**Primary metric: Precision@K (specifically Precision@20 and Precision@50)**

**What it measures:**
Of the top K pages the model ranks as highest-priority refresh candidates,
what fraction are actually declining?

**Formula:**
Precision@K = (number of truly declining pages in top K) / K

**Why this metric fits the business decision:**
A content team has limited writer capacity. They will act on the top N pages
from the ranked queue each week. Precision@K directly measures what they care
about: are the pages I am spending time on actually worth fixing?

**Baseline to beat (from Week 2):**
- Hand rule Precision@20 = 0.900
- Hand rule Precision@50 = 0.680

**What "good" means for the capstone model:**
- Precision@50 ≥ 0.700 on the held-out validation set (beats the hand rule
  at K=50, the more practically relevant cutoff for a real content team)
- The model must also flag pages BEYOND the 17 the hand rule catches —
  otherwise it adds no value over the existing rule

**Secondary metric: Coverage of declining pages**
The hand rule catches only 17 of the ~16,260 declining pages (0.1%).
A secondary success criterion is that the model's top-200 recommendations
cover a meaningfully larger fraction of the true declining population.

**What the metric cannot tell us:**
Precision@K measures ranking quality on this dataset. It does not measure
whether refreshing a flagged page will actually improve its performance.
That causal claim requires a controlled experiment, which is outside the
scope of this capstone.

In [4]:
# Demonstrate the Precision@K metric on the Week 2 hand rule baseline
# This anchors the success metric with real numbers

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Recreate the Week 2 hand rule
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

y = df["is_declining"].values

print("Baseline performance (Week 2 hand rule):")
print("=" * 45)
for k in [10, 20, 50, 100]:
    p = precision_at_k(df["hand_rule_score"], y, k)
    print(f"  Precision@{k:<4} = {p:.3f}   "
          f"({round(p*k)}/{k} pages correctly flagged)")

print()
print(f"Pages flagged by hand rule (score > 0): "
      f"{(df['hand_rule_score'] > 0).sum():,}")
print(f"Total declining pages in dataset:       "
      f"{y.sum():,}")
print(f"Coverage: hand rule catches "
      f"{(df['hand_rule_score'] > 0).sum() / y.sum() * 100:.1f}% "
      f"of all declining pages")
print()
print("Target: capstone model Precision@50 ≥ 0.700 on held-out split")
print("        with broader coverage than the 17-page hand rule queue")

Baseline performance (Week 2 hand rule):
  Precision@10   = 1.000   (10/10 pages correctly flagged)
  Precision@20   = 0.900   (18/20 pages correctly flagged)
  Precision@50   = 0.680   (34/50 pages correctly flagged)
  Precision@100  = 0.630   (63/100 pages correctly flagged)

Pages flagged by hand rule (score > 0): 17
Total declining pages in dataset:       16,262
Coverage: hand rule catches 0.1% of all declining pages

Target: capstone model Precision@50 ≥ 0.700 on held-out split
        with broader coverage than the 17-page hand rule queue


## 4. The unit of analysis, as a real dataframe

**One row = one content page, observed over a 90-day window.**

Each row represents a single published web page and its aggregated search
performance signals over the most recent 90-day measurement period. The page
is the decision unit: a content team decides to refresh (or not) at the
page level, and the model scores at the page level.

The relevant columns for Lane 2 are:

| Column | Type | Role |
|---|---|---|
| `content_id` | identifier | unique page ID |
| `client_id` | identifier | which client owns the page |
| `days_since_last_update` | feature | staleness signal |
| `content_age_days` | feature | how old the page is |
| `impressions_90d` | feature | visibility over 90 days |
| `avg_position` | feature | average search rank |
| `ctr` | feature | click-through rate |
| `word_count` | feature | content length |
| `engagement_rate` | feature | user engagement |
| `trend_direction` | label source | used to build proxy label |
| `trend_pct` | EXCLUDED | leaky — is the label in numeric form |
| `is_declining` | target | proxy label we defined (0 or 1) |

The code cell below shows exactly what one row looks like, and what the
target column looks like alongside the key features.

In [5]:
# Show the unit of analysis as a real dataframe slice
# One row = one page

feature_cols = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "content_age_days",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate",
    "trend_direction",
    "is_declining"          # our proxy label
]

# Show a sample of 8 rows — mix of declining and not declining
sample_declining = df[df["is_declining"] == 1].head(4)
sample_not = df[df["is_declining"] == 0].head(4)
sample = pd.concat([sample_declining, sample_not])[feature_cols].reset_index(drop=True)

print("Unit of analysis: one row = one content page (90-day window)")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Lane 2 uses {len(feature_cols)-2} features + 1 target column\n")
print(sample.to_string())

print()
print("Target column distribution in this sample:")
print(sample["is_declining"].value_counts().to_string())
print()
print("is_declining = 1 → page is a refresh candidate")
print("is_declining = 0 → page is not currently flagged")

Unit of analysis: one row = one content page (90-day window)
Dataset shape: 30,000 rows × 46 columns
Lane 2 uses 9 features + 1 target column

             content_id          client_id  days_since_last_update  content_age_days  impressions_90d  avg_position   ctr  word_count  engagement_rate trend_direction  is_declining
0  content_304f48230142  client_f369cb89fc                      20               187             3803          10.6  0.76      3221.0             5.88            down             1
1  content_a1fb4e703a9e  client_4e07408562                      25               445            15320          20.3  0.05      2481.0             0.00            down             1
2  content_9aa793d4d895  client_7f2253d7e2                      20               141            12581          36.5  0.09      3515.0             0.00            down             1
3  content_d99b7a2d90ca  client_3fdba35f04                      14               263            19140          44.0  0.13      2803.0

## 5. Why ML beats a fixed rule here

**The core problem with fixed rules: they cannot handle interacting signals.**

The Week 2 hand rule uses exactly two conditions:
- `days_since_last_update ≥ 180`
- `impressions_90d ≥ 500`

This catches 17 pages. But 16,260 pages are declining. The rule misses 99.9%
of the problem — not because it is wrong, but because it is too rigid.

**Three specific reasons ML is needed here:**

**1. Declining pages do not share a single profile.**
A page can be declining because it is old and stale (the rule catches this).
But it can also be declining because its CTR is dropping relative to similar-
ranked pages, or because engagement is falling even while impressions hold
steady. These are different patterns. A fixed rule cannot combine them. A
model trained on multiple features can.

**2. The signals interact in non-obvious ways.**
A page with high impressions and low CTR at position 3 is a very different
problem from a page with low impressions and low CTR at position 15. The
combination of avg_position + ctr + impressions_90d tells a story that no
single threshold captures. This is exactly what a learned model is designed
to do: find the combinations of features that predict the outcome.

**3. The thresholds are arbitrary.**
Why 180 days? Why 500 impressions? These numbers were chosen by intuition.
A model learns the threshold from the data itself — and it can find that
the real signal is different from what intuition suggested. Week 2 already
showed this: when the decision tree was given the same features, it found
`avg_position` as the first split, not `days_since_last_update`. The data
disagreed with the human intuition.

**What the model will do that the rule cannot:**
- Combine 6+ signals into a single continuous score
- Assign different weights to each signal based on what the data shows
- Score ALL 30,000 pages, not just the 17 that pass both thresholds
- Generalize to new pages from held-out clients

**The honest caveat:**
ML beats the fixed rule at *coverage* and *generalization*. It does not
guarantee that the flagged pages actually need a refresh in the causal sense.
The model is a decision-support tool, not a ground truth oracle.

In [6]:
# Show concretely why a fixed rule misses real declining pages
# by looking at declining pages that the hand rule does NOT catch

missed_by_rule = df[(df["is_declining"] == 1) & (df["hand_rule_score"] == 0)]
caught_by_rule = df[(df["is_declining"] == 1) & (df["hand_rule_score"] > 0)]

print("Declining pages caught vs missed by the hand rule:")
print("=" * 55)
print(f"  Caught by hand rule:  {len(caught_by_rule):>6,} pages")
print(f"  Missed by hand rule:  {len(missed_by_rule):>6,} pages")
print(f"  Miss rate:            {len(missed_by_rule)/df['is_declining'].sum()*100:.1f}%")

print()
print("Profile of missed declining pages (median values):")
print("-" * 55)
cols_to_show = ["impressions_90d", "avg_position", "ctr",
                "days_since_last_update", "engagement_rate"]
medians = missed_by_rule[cols_to_show].median()
for col, val in medians.items():
    print(f"  {col:<30} {val:.2f}")

print()
print("These pages are REAL declining pages with real impressions.")
print("They are invisible to the hand rule because they do not")
print("meet both thresholds simultaneously.")
print("A scoring model trained on multiple signals can surface them.")

Declining pages caught vs missed by the hand rule:
  Caught by hand rule:      16 pages
  Missed by hand rule:  16,246 pages
  Miss rate:            99.9%

Profile of missed declining pages (median values):
-------------------------------------------------------
  impressions_90d                960.00
  avg_position                   11.30
  ctr                            0.08
  days_since_last_update         20.00
  engagement_rate                0.00

These pages are REAL declining pages with real impressions.
They are invisible to the hand rule because they do not
meet both thresholds simultaneously.
A scoring model trained on multiple signals can surface them.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card

---

**Task type confirmed:** Scoring → Ranking (pointwise)

**Proxy label:** `is_declining = (trend_direction == "down")`
— reasonable but imperfect; limitations acknowledged

**Success metric:** Precision@50 ≥ 0.700 on client-holdout validation split
— baseline to beat: hand rule Precision@50 = 0.680

**Unit of analysis:** One row = one content page, 90-day observation window

**Key finding:** The hand rule misses 99.9% of declining pages (16,243 of 16,260).
A scoring model combining 6+ signals can surface this population.
The gap between rule coverage and true declining rate is the ML opportunity.

**Next step:** w03 — data contract and feature audit